# Encoder Pipeline — Full FCMAE Pretraining (+ cross-view) — HPC training twin

Full-scale GPU run of the encoder pretraining, mirroring
`notebooks/modeling/01_encoder_pipeline.ipynb`. Replaces SimCLR with **FCMAE masked-region
pretraining** (Stage P1) and **cross-view completion** (Stage P2) — the pretext task ConvNeXtV2 was
co-designed for. FCMAE reconstructs masked patches, forcing *local-geometry* modelling with **no**
invariance pressure on the fracture signal; labels are **monitoring-only** so the stage stays
label-free and pretrained-once globally. The architecture / masking / decoder code is **verbatim**
from the local notebook; only the run scaffolding differs (CUDA, AMP, warmup→cosine over many epochs,
checkpoint-every-N + resume, early-stop on the monitor loss).

**Curriculum:** P1 (FCMAE, from the ImageNet-FCMAE ConvNeXtV2 checkpoint) → P2 (joint FCMAE +
cross-view completion, from P1). P1 and P2 are exported as **separate** encoder checkpoints; the
blessed `convnextv2_fcmae_encoder.pth` is the contract file the decoder loads. Gate G4 (downstream)
decides which pretraining, if any, ships.

Quality signals are **M-6** (representation probe vs the ImageNet-init reference) and **Gate G4**
(downstream) — not a contrastive-loss value. NT-Xent is retired.

### How to run (HPC)

```bash
cd "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059"
source .venv/bin/activate
pip install -r requirements.txt
```

Run the cells top to bottom; the **CONFIG** cell is the only place you change settings. Set
`SMOKE_TEST = True` first for a quick CPU-friendly dry-run of the whole curriculum before the GPU run.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce CUDA fragmentation (OOM safeguard); must be set before torch initialises CUDA
import math, random, time, shutil, json
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.ndimage as ndi
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import timm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# ============================= CONFIG (HPC - full FCMAE pretraining) =============================
# Mirrors the local encoder notebook; only these knobs differ. Run on the Sunway HPC GPU.
ENV          = "HPC"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device       = DEVICE                     # alias: the verbatim FCMAE building blocks reference `device`
IMG_SIZE     = 256
# --- curriculum epochs (P1 to plateau, then P2 joint) ---
P1_EPOCHS    = 250          # FCMAE adaptation on a small dataset happens early; 100-400 with early stop
P2_EPOCHS    = 100          # cross-view stage, from the P1 encoder
RUN_P1, RUN_P2 = True, True
BATCH_SIZE   = 16           # micro-batch that fits 22GB at 256^2 (was 128, OOM); effective batch = BATCH_SIZE*ACCUM_STEPS
ACCUM_STEPS  = 8            # gradient accumulation -> effective batch 128 (preserves the FCMAE recipe)
LR           = 1.5e-4       # peak LR is scaled by (BATCH_SIZE*ACCUM_STEPS/256) in the loop (FCMAE recipe)
WARMUP_FRAC  = 0.08         # linear warmup over this fraction of steps, then cosine decay
WEIGHT_DECAY = 0.05
NUM_WORKERS  = 4
USE_AMP      = True
USE_GRAD_CKPT = True        # gradient-checkpoint the backbone (memory saver, ~30% slower; was False)
CKPT_EVERY   = 25
EARLY_STOP_PATIENCE = 30    # stop a stage if the monitor loss has not improved for this many epochs
PRETRAINED   = True         # ImageNet-FCMAE ConvNeXtV2 init (also the Gate-G4c control)
FREEZE_ENCODER = False
INCLUDE_GEOMETRIC = True    # SSL has no 3D GT, so geometric variants are safe extra views
# --- FCMAE / cross-view hyperparameters (identical to the local notebook) ---
PATCH        = 32
MASK_GRID    = IMG_SIZE // PATCH
MASK_RATIO   = 0.6
OCC_ELIGIBLE = 0.05
BG_QUOTA     = 0.15
SIGMA_FLOOR  = 0.01
MIN_ELIGIBLE = 8
STRUCT_WEIGHT = 2.0
MONITOR_FRAC = 0.10
LAMBDA_XVIEW = 1.0
XVIEW_SLACK  = 1
# --- defect-preserving augmentation only (§2.5); FORBIDDEN: cutout/erasing/aggressive-crop/elastic ---
AUG_HFLIP    = False
AUG_ROT_DEG  = 8.0
AUG_TRANS    = 0.05
AUG_GAMMA    = (0.85, 1.15)
# --- smoke / resume ---
SMOKE_TEST   = False
SMOKE_EPOCHS = 2
SMOKE_STEPS  = 10
SMOKE_CASES_PER_GROUP = 3
RESUME_P1    = None         # e.g. str(MODELS_DIR / "fcmae_p1_trainstate.pth")
RESUME_P2    = None
EXPLICIT_ROOT = None        # e.g. "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059"
if DEVICE.type != "cuda":
    print("[warning] CUDA not available - this HPC notebook expects a GPU (CPU is fine only for SMOKE_TEST).")
print("ENV", ENV, "| device", DEVICE, "| P1 epochs", P1_EPOCHS, "| P2 epochs", P2_EPOCHS,
      "| batch", BATCH_SIZE, "| accum", ACCUM_STEPS, "| grad_ckpt", USE_GRAD_CKPT, "| smoke", SMOKE_TEST)

In [ ]:
# Resolve the project root robustly (works locally and on HPC, regardless of where the notebook is
# launched from). We look upward for data/interim/predrr (the GT CT folder), matching
# 03_decoder_pipeline.ipynb so both notebooks resolve the same ROOT.
def find_root(start: Path) -> Path:
    if EXPLICIT_ROOT:
        r = Path(EXPLICIT_ROOT)
        if (r / "data" / "interim" / "predrr").exists():
            return r
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("Could not find project root (expected data/interim/predrr). "
                            "Set EXPLICIT_ROOT in the CONFIG cell.")

ROOT            = find_root(Path.cwd())
DATA            = ROOT / "data"
NORMAL_DRR_DIR  = DATA / "interim" / "DRRs"                 # normal AP/LAT DRRs
AUG_DRR_DIR     = DATA / "processed" / "augmented_DRRs"     # augmented DRR variants
PREDRR_DIR      = DATA / "interim" / "predrr"               # GT CT (only anchors ROOT here)
MODELS_DIR      = ROOT / "models"; MODELS_DIR.mkdir(exist_ok=True)
# FCMAE checkpoints: per-stage encoders + the blessed contract file the decoder loads
FCMAE_P1_CKPT      = MODELS_DIR / "convnextv2_fcmae_p1_encoder.pth"
FCMAE_P2_CKPT      = MODELS_DIR / "convnextv2_fcmae_p2_encoder.pth"
FCMAE_CKPT         = MODELS_DIR / "convnextv2_fcmae_encoder.pth"       # <- the contract file the decoder loads
MONITOR_SPLIT_CSV  = MODELS_DIR / "pretrain_monitor_split.csv"
P1_HISTORY_CSV     = MODELS_DIR / "fcmae_p1_history.csv"
P2_HISTORY_CSV     = MODELS_DIR / "fcmae_p2_history.csv"
P1_TRAINSTATE      = MODELS_DIR / "fcmae_p1_trainstate.pth"            # full state for RESUME_P1
P2_TRAINSTATE      = MODELS_DIR / "fcmae_p2_trainstate.pth"            # full state for RESUME_P2
print("ROOT:", ROOT)
print("blessed encoder checkpoint ->", FCMAE_CKPT)

## 1. Shared encoder front-end + intensity structure mask

The encoder front-end below is **copied verbatim** from `01_encoder_pipeline.ipynb` / the decoder
twin, so the `encoder.state_dict()` this notebook saves loads back into `BiPlanarFeatureFusion`
with `missing=0 unexpected=0`. **Do not edit it here.** The blessed FCMAE checkpoint is a bare
`state_dict` with matching keys, loaded via `fusion.load_pretrained_encoder(...)`.

We also define `_soft_bone_mask` — the **intensity** structure mask (2D analogue of predrr's
`body_envelope_mask`) — because the FCMAE masking (§2) restricts masking to the structure region.
**Provenance (L-3):** this mask is derived from DRR *intensity*, never from the GT surface meshes, so
using it to drive masking is not label leakage.

In [ ]:
# ===== Encoder front-end - VERBATIM from 01_encoder_pipeline.ipynb. DO NOT EDIT. =====
BACKBONE     = "convnextv2_tiny"
OUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attn", "attn"]   # fine -> coarse

def make_backbone(pretrained=True):
    """features_only ConvNeXtV2 returning 4 multi-scale maps. Falls back to random init offline."""
    try:
        return timm.create_model(BACKBONE, pretrained=pretrained, features_only=True)
    except Exception as e:
        print("[warn] pretrained fetch failed (%s); random init." % type(e).__name__)
        return timm.create_model(BACKBONE, pretrained=False, features_only=True)

FEAT_DIMS = [f["num_chs"] for f in make_backbone(pretrained=False).feature_info]   # [96,192,384,768]

def load_drr(path):
    """npy 256x256 float32 [0,1] -> tensor [3,H,W] (1 channel replicated to 3 for ConvNeXtV2)."""
    arr = np.load(path).astype(np.float32)
    t = torch.from_numpy(arr)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return t.repeat(3, 1, 1) if t.shape[0] == 1 else t

NORMALIZE = T.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
def paired_tf(t):
    return NORMALIZE(t)

class CrossAttention(nn.Module):
    """AP (query) attends to LAT (key/value). Operates on tokens [B, N, C]."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, a, b):
        attn = F.softmax(torch.matmul(self.q(a), self.k(b).transpose(-2, -1)) * self.scale, dim=-1)
        return torch.matmul(attn, self.v(b)) + a

class LocalFusion(nn.Module):
    """Cheap high-res fusion: concat views + 3x3 conv, residual on AP."""
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, kernel_size=3, padding=1)
    def forward(self, a, b):
        return self.mix(torch.cat([a, b], dim=1)) + a

# --- bi-planar lift orientation (resolved empirically; see Check 1 / _axis_probe) ---
# GT array axes, from nibabel axcodes ('L','P','S'): axis0 = L-R, axis1 = A-P, axis2 = S-I.
# AP projects along A-P (axis1); LAT along L-R (axis0). Both DRR rows (H) = S-I (axis2);
# AP cols (W) = L-R (axis0); LAT cols (W) = A-P (axis1). Cube ordered (axis0,axis1,axis2) to
# match the GT array. flip_* reverse a row/col vs its volume axis; confirmed by overfit guard.
LIFT_FLIP_SI      = False   # DRR rows  vs axis2 (S-I)
LIFT_FLIP_AP_COL  = False   # AP  cols  vs axis0 (L-R)
LIFT_FLIP_LAT_COL = True    # LAT cols  vs axis1 (A-P)

class BiPlanarFeatureFusion(nn.Module):
    def __init__(self, feat_dims=FEAT_DIMS, out_channels=OUT_CHANNELS,
                 fusion_types=FUSION_TYPES, depth=16, pretrained=True, freeze_encoder=False):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        self.fusion_types = list(fusion_types); self.depth = depth
        self.fuse = nn.ModuleList([CrossAttention(d) if t == "attn" else LocalFusion(d)
                                   for d, t in zip(feat_dims, fusion_types)])
        self.to3d = nn.ModuleList([nn.Conv2d(c, o, 1) for c, o in zip(feat_dims, out_channels)])
        self.expand3d = nn.ModuleList([nn.Conv3d(2 * o, o, 3, padding=1) for o in out_channels])
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
    def load_pretrained_encoder(self, path):
        """Load a bare encoder state_dict (the FCMAE blessed checkpoint) into the SAME features_only
        ConvNeXtV2 backbone. missing=0 expected — the export writes matching keys."""
        missing, unexpected = self.encoder.load_state_dict(torch.load(path, map_location="cpu"), strict=False)
        print("loaded pretrained encoder: missing=%d unexpected=%d" % (len(missing), len(unexpected)))
        return missing, unexpected
    def _ortho_lift(self, ap_f, lat_f, c2d, c3d):
        """Orthogonal back-projection lift: place each view on the two volume axes it resolves
        and broadcast along its unobserved projection axis, then fuse the two cubes in 3D.
        Output cube ordered (axis0=L-R, axis1=A-P, axis2=S-I) to match the GT array."""
        B, C, H, W = ap_f.shape           # square feature map at this level: S = H = W
        S = H
        ap = c2d(ap_f); lat = c2d(lat_f)  # shared 1x1 projection -> [B, O, H, W] each
        O = ap.shape[1]
        if LIFT_FLIP_SI:      ap = ap.flip(2); lat = lat.flip(2)   # rows (H) = S-I (axis2)
        if LIFT_FLIP_AP_COL:  ap = ap.flip(3)                       # AP  cols (W) = L-R (axis0)
        if LIFT_FLIP_LAT_COL: lat = lat.flip(3)                     # LAT cols (W) = A-P (axis1)
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(B, O, S, S, S)   # broadcast axis1 (A-P)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(B, O, S, S, S)  # broadcast axis0 (L-R)
        return c3d(torch.cat([ap_cube, lat_cube], dim=1))           # [B, O, S, S, S]
    def forward(self, ap_img, lat_img):
        ap_feats, lat_feats = self.encoder(ap_img), self.encoder(lat_img)
        fused2d, fused3d = [], []
        for ap_f, lat_f, fuse, c2d, c3d, t in zip(
                ap_feats, lat_feats, self.fuse, self.to3d, self.expand3d, self.fusion_types):
            B, C, H, W = ap_f.shape
            if t == "attn":
                a = ap_f.flatten(2).transpose(1, 2); b = lat_f.flatten(2).transpose(1, 2)
                f2d = fuse(a, b).transpose(1, 2).reshape(B, C, H, W)
            else:
                f2d = fuse(ap_f, lat_f)
            fused2d.append(f2d)
            fused3d.append(self._ortho_lift(ap_f, lat_f, c2d, c3d))
        return fused2d, fused3d

# ===== intensity structure mask (VERBATIM §1b from 01_encoder_pipeline.ipynb) — drives FCMAE masking =====
BONE_THR, BONE_MIN_AREA, BONE_MARGIN_PX, BONE_FEATHER_SIG = 0.40, 0, 6, 2.0
def _soft_bone_mask(arr2d):
    """HxW float32 [0,1], bg=0 -> [0,1] feathered bone mask (intensity-derived, NOT GT). No distance
    limit so a displaced fragment survives even when far from the main bone."""
    bone = arr2d > BONE_THR
    if BONE_MIN_AREA > 0:
        lbl, n = ndi.label(bone)
        if n:
            sizes = ndi.sum(np.ones_like(lbl), lbl, range(1, n + 1))
            bone = np.isin(lbl, 1 + np.where(sizes >= BONE_MIN_AREA)[0])
    bone = ndi.binary_fill_holes(bone)
    bone = ndi.binary_dilation(bone, iterations=BONE_MARGIN_PX)
    return np.clip(ndi.gaussian_filter(bone.astype(np.float32), BONE_FEATHER_SIG), 0, 1)

print("encoder feature dims:", FEAT_DIMS, "| structure mask ready")

## 2. Data — all-knee DRRs + label-free monitor split

We index both normal and augmented DRRs. FCMAE pretrains on the **gradient split** of all knees; a
held-out **monitor set** (≈`MONITOR_FRAC` of instances, both views together, stratified by a
**label-free fragmentation proxy** = connected-component count of the intensity structure mask) is
excluded from all pretraining gradients so the §5 monitor curves are real (L-4). The split is written
to `pretrain_monitor_split.csv`, keyed like the fold lists, so it is reproducible.

> **Cross-validation note.** FCMAE uses **no occupancy labels**, so it pretrains **once on the
> gradient split of all knees** and the single fold-agnostic encoder is reused across folds — no
> **label** leakage in the comparison; only the standard, documented *representation* exposure to test
> *images* remains. Labels (healthy/fractured) are used for **monitoring only** (M-3/M-6, eval-side).

In [ ]:
def build_paired_index():
    """One row per (case, side, variant) with absolute AP/LAT paths + metadata."""
    rows = []
    nmeta = pd.read_csv(NORMAL_DRR_DIR / "drr_generation_metadata.csv")
    for (ds, case, side), _ in nmeta.groupby(["dataset", "case", "side"]):
        ap = NORMAL_DRR_DIR / ds / case / side / "ap.npy"
        lat = NORMAL_DRR_DIR / ds / case / side / "lat.npy"
        if ap.exists() and lat.exists():
            rows.append(dict(dataset=ds, case=case, side=side, variant="normal",
                             geometric=False, ap=str(ap), lat=str(lat)))
    ameta_path = AUG_DRR_DIR / "augmentation_variants_metadata.csv"
    if ameta_path.exists():
        ameta = pd.read_csv(ameta_path)
        for r in ameta.itertuples(index=False):
            ap = AUG_DRR_DIR / r.ap_npy; lat = AUG_DRR_DIR / r.lat_npy
            if ap.exists() and lat.exists():
                rows.append(dict(dataset=r.dataset, case=r.case, side=r.side, variant=r.variant,
                                 geometric=bool(r.geometric), ap=str(ap), lat=str(lat)))
    return pd.DataFrame(rows)

paired_index = build_paired_index()
if not INCLUDE_GEOMETRIC:
    paired_index = paired_index[~paired_index.geometric].reset_index(drop=True)
if SMOKE_TEST:
    _keep = paired_index.groupby("dataset").head(SMOKE_CASES_PER_GROUP * 4)
    paired_index = _keep.reset_index(drop=True)

# --- label-free monitor split (L-4): hold out ~MONITOR_FRAC of INSTANCES, stratified by the ---
# connected-component count of the intensity mask (image-derived, NO cohort label, L-2). Excluded
# from all pretraining gradients only; still used in downstream fold training as normal.
def fragmentation_proxy(ap_path):
    arr = np.load(ap_path).astype(np.float32)
    _, n = ndi.label(_soft_bone_mask(arr) > 0.5)
    return int(n)

_inst = (paired_index.groupby(["dataset", "case", "side"]).agg(ap=("ap", "first")).reset_index())
_inst["frag"] = _inst["ap"].map(fragmentation_proxy)
_rng = np.random.default_rng(SEED)
_inst["bin"] = pd.qcut(_inst["frag"].rank(method="first"), q=min(3, len(_inst)), labels=False)
_mon_idx = set()
for _b, g in _inst.groupby("bin"):
    k = max(1, int(round(MONITOR_FRAC * len(g))))
    _mon_idx.update(_rng.choice(g.index.to_numpy(), size=min(k, len(g)), replace=False).tolist())
_inst["role"] = np.where(_inst.index.isin(_mon_idx), "monitor", "gradient")
_inst[["dataset", "case", "side", "frag", "role"]].to_csv(MONITOR_SPLIT_CSV, index=False)
MONITOR_INSTANCES = set(map(tuple, _inst.loc[_inst.role == "monitor", ["dataset", "case", "side"]].values))

def _is_monitor(r):
    return (r["dataset"], r["case"], r["side"]) in MONITOR_INSTANCES
grad_index = paired_index[~paired_index.apply(_is_monitor, axis=1)].reset_index(drop=True)
monitor_index = paired_index[paired_index.apply(_is_monitor, axis=1)].reset_index(drop=True)

print("paired rows:", len(paired_index), "| instances:", len(_inst), "| monitor:", len(MONITOR_INSTANCES))
print("gradient pairs:", len(grad_index), "| monitor pairs:", len(monitor_index))
print("cases:", paired_index.groupby("dataset")["case"].nunique().to_dict(), "| saved:", MONITOR_SPLIT_CSV.name)

## 3. FCMAE building blocks (verbatim from `01_encoder_pipeline.ipynb`)

Masked encoder (dense per-block re-zeroing so the encoder cannot peek across the mask boundary; GRN
stays active), tiny decoder, structure-restricted masking driven by the intensity mask, per-patch
normalized targets with a σ-floor, masked loss, and **defect-preserving** augmentation only
(rotation/translate/gamma; **forbidden:** cutout/erasing/aggressive-crop/elastic). Identical to the
local notebook so the exported encoder is byte-compatible.

In [ ]:
# ============================ FCMAE building blocks (verbatim from local §2.1) ============================

class FCMAEEncoder(nn.Module):
    """Masked ConvNeXtV2: re-zero the mask after the stem AND after every block (dense FCMAE, §2.2)."""
    def __init__(self, pretrained=PRETRAINED):
        super().__init__()
        self.backbone = make_backbone(pretrained)
    def _vis(self, x, mask_grid):
        return x * F.interpolate((~mask_grid).float(), size=x.shape[-2:], mode="nearest")
    def forward(self, x, mask_grid, rezero=True):
        bb = self.backbone
        if rezero: x = self._vis(x, mask_grid)
        x = bb.stem_1(bb.stem_0(x))
        if rezero: x = self._vis(x, mask_grid)
        feats = []
        for i in range(4):
            st = getattr(bb, f"stages_{i}")
            x = st.downsample(x)
            if rezero: x = self._vis(x, mask_grid)
            for blk in st.blocks:
                x = blk(x)
                if rezero: x = self._vis(x, mask_grid)
            feats.append(x)
        return feats

class ConvNeXtLite(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dw = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)
        self.norm = nn.GroupNorm(1, dim)
        self.pw1 = nn.Conv2d(dim, 4 * dim, 1); self.act = nn.GELU(); self.pw2 = nn.Conv2d(4 * dim, dim, 1)
    def forward(self, x):
        r = x; x = self.dw(x); x = self.norm(x); x = self.pw2(self.act(self.pw1(x)))
        return x + r

class TinyFCMAEDecoder(nn.Module):
    def __init__(self, in_dim=None, dim=512, patch=PATCH):
        super().__init__()
        in_dim = in_dim or FEAT_DIMS[-1]
        self.proj_in = nn.Conv2d(in_dim, dim, 1)
        self.block = ConvNeXtLite(dim)
        self.to_pixels = nn.Conv2d(dim, patch * patch, 1)
        self.patch = patch
    def forward(self, feat32):
        z = self.block(self.proj_in(feat32))
        px = self.to_pixels(z)
        B = px.shape[0]
        px = px.view(B, self.patch, self.patch, MASK_GRID, MASK_GRID)
        return px.permute(0, 3, 4, 1, 2).unsqueeze(1)   # [B,1,gh,gw,ph,pw]

# PROVENANCE (L-3): masking eligibility uses _soft_bone_mask (intensity), NEVER a GT-derived mask.
assert "_soft_bone_mask" in globals(), "structure mask must be defined before FCMAE masking (L-3)"

def patch_occupancy(arr2d):
    binm = (_soft_bone_mask(arr2d) > 0.5).astype(np.float32)
    return binm.reshape(MASK_GRID, PATCH, MASK_GRID, PATCH).mean(axis=(1, 3))

def sample_mask(occ):
    struct = occ >= OCC_ELIGIBLE
    E = struct.copy()
    bg_idx = np.flatnonzero((~struct).ravel())
    nq = int(round(BG_QUOTA * max(1, struct.sum())))
    if len(bg_idx):
        pick = np.random.choice(bg_idx, size=min(nq, len(bg_idx)), replace=False)
        Ef = E.ravel(); Ef[pick] = True; E = Ef.reshape(MASK_GRID, MASK_GRID)
    idx = np.flatnonzero(E.ravel())
    mask = np.zeros(MASK_GRID * MASK_GRID, dtype=bool)
    if len(idx) >= MIN_ELIGIBLE:
        n = max(1, int(round(MASK_RATIO * len(idx))))
        mask[np.random.choice(idx, size=n, replace=False)] = True
    return mask.reshape(MASK_GRID, MASK_GRID), struct, len(idx)

def build_batch_masks(imgs_raw):
    masks, structs, skip = [], [], []
    for b in range(imgs_raw.shape[0]):
        occ = patch_occupancy(imgs_raw[b, 0].cpu().numpy())
        m, s, nE = sample_mask(occ)
        masks.append(torch.from_numpy(m)); structs.append(torch.from_numpy(s)); skip.append(nE < MIN_ELIGIBLE)
    return torch.stack(masks).unsqueeze(1), torch.stack(structs).unsqueeze(1), torch.tensor(skip)

def per_patch_normalize(img1ch, sigma_floor=SIGMA_FLOOR):
    p = img1ch.unfold(2, PATCH, PATCH).unfold(3, PATCH, PATCH)
    mu = p.mean(dim=(-1, -2), keepdim=True); sd = p.std(dim=(-1, -2), keepdim=True)
    return (p - mu) / sd.clamp_min(sigma_floor), sd

def fcmae_loss(pred, target, mask_grid, struct_grid=None, struct_weight=STRUCT_WEIGHT):
    m = mask_grid.unsqueeze(-1).unsqueeze(-1).float()
    w = torch.ones_like(pred)
    if struct_grid is not None:
        w = 1.0 + (struct_weight - 1.0) * struct_grid.unsqueeze(-1).unsqueeze(-1).float()
    se = (pred - target) ** 2
    return (se * w * m).sum() / (w * m).sum().clamp_min(1.0)

def sample_aug_params():
    return dict(angle=random.uniform(-AUG_ROT_DEG, AUG_ROT_DEG),
                tx=int(random.uniform(-AUG_TRANS, AUG_TRANS) * IMG_SIZE),
                ty=int(random.uniform(-AUG_TRANS, AUG_TRANS) * IMG_SIZE),
                gamma=random.uniform(*AUG_GAMMA),
                flip=AUG_HFLIP and random.random() < 0.5)

def apply_aug(img1ch, p):
    if p["flip"]:
        img1ch = TF.hflip(img1ch)
    img1ch = TF.affine(img1ch, angle=p["angle"], translate=[p["tx"], p["ty"]], scale=1.0, shear=[0.0, 0.0])
    return img1ch.clamp(0, 1).pow(p["gamma"]).clamp(0, 1)

NORM_MEAN, NORM_STD = 0.5, 0.5
def to_encoder_input(img1ch):
    return ((img1ch - NORM_MEAN) / NORM_STD).repeat(1, 3, 1, 1)

class FCMAEAnchorDataset(Dataset):
    def __init__(self, df):
        recs = []
        for r in df.itertuples(index=False):
            recs.append((r.ap, r.dataset, r.case, r.side, "ap"))
            recs.append((r.lat, r.dataset, r.case, r.side, "lat"))
        self.recs = recs
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        path, ds, case, side, view = self.recs[i]
        return torch.from_numpy(np.load(path).astype(np.float32)).unsqueeze(0), ds, case, side, view

class FCMAEPairedDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ap = torch.from_numpy(np.load(r.ap).astype(np.float32)).unsqueeze(0)
        lat = torch.from_numpy(np.load(r.lat).astype(np.float32)).unsqueeze(0)
        return ap, lat, r.dataset, r.case, r.side, r.variant

def run_fcmae_batch(enc, dec, imgs_raw, augment=True):
    if augment:
        imgs_raw = torch.stack([apply_aug(imgs_raw[b], sample_aug_params()) for b in range(imgs_raw.shape[0])])
    imgs_raw = imgs_raw.to(device)
    mg, sg, skip = build_batch_masks(imgs_raw); mg, sg = mg.to(device), sg.to(device)
    feats = enc(to_encoder_input(imgs_raw), mg)
    pred = dec(feats[-1])
    tgt, sd = per_patch_normalize(imgs_raw)
    return fcmae_loss(pred, tgt, mg, sg), dict(sd=sd, mask=mg, struct=sg, skip=skip)

def rowwise_attn_mask(grid=MASK_GRID, slack=XVIEW_SLACK, dev=None):
    idx = torch.arange(grid * grid)
    ri, rj = (idx // grid).view(-1, 1), (idx // grid).view(1, -1)
    m = torch.zeros(grid * grid, grid * grid)
    m.masked_fill_((ri - rj).abs() > slack, float("-inf"))
    return m.to(dev or device)

class CrossViewBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim); self.self_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ln2 = nn.LayerNorm(dim); self.cross_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ln3 = nn.LayerNorm(dim); self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
    def forward(self, tgt, src, attn_mask):
        t = self.ln1(tgt); tgt = tgt + self.self_attn(t, t, t, need_weights=False)[0]
        t = self.ln2(tgt); tgt = tgt + self.cross_attn(t, src, src, attn_mask=attn_mask, need_weights=False)[0]
        return tgt + self.mlp(self.ln3(tgt))

class CrossViewDecoder(nn.Module):
    def __init__(self, dim=None, depth=2, patch=PATCH):
        super().__init__()
        dim = dim or FEAT_DIMS[-1]
        self.blocks = nn.ModuleList([CrossViewBlock(dim) for _ in range(depth)])
        self.head = nn.Linear(dim, patch * patch); self.patch = patch
    def forward(self, tgt_feat32, src_feat32, attn_mask):
        B = tgt_feat32.shape[0]
        tgt = tgt_feat32.flatten(2).transpose(1, 2); src = src_feat32.flatten(2).transpose(1, 2)
        for blk in self.blocks:
            tgt = blk(tgt, src, attn_mask)
        px = self.head(tgt).transpose(1, 2).view(B, self.patch, self.patch, MASK_GRID, MASK_GRID)
        return px.permute(0, 3, 4, 1, 2).unsqueeze(1)

def crossview_loss(enc, xdec, tgt_raw, src_raw, attn_mask):
    tgt_raw = tgt_raw.to(device); src_raw = src_raw.to(device)
    mg, sg, _ = build_batch_masks(tgt_raw); mg, sg = mg.to(device), sg.to(device)
    tgt_feat = enc(to_encoder_input(tgt_raw), mg)[-1]
    src_feat = enc(to_encoder_input(src_raw), torch.zeros_like(mg))[-1]
    pred = xdec(tgt_feat, src_feat, attn_mask)
    tgt, _ = per_patch_normalize(tgt_raw)
    return fcmae_loss(pred, tgt, mg, sg)

print("FCMAE building blocks ready:", "encoder+decoder+masking+loss+crossview")

## 4. FCMAE pretraining — full curriculum with checkpointing

`AdamW` (β=0.9,0.95, wd=`WEIGHT_DECAY`), peak LR `1.5e-4 × (batch/256)`, linear **warmup → cosine**
decay per stage; mixed precision on GPU (bf16 where available, else fp16 + GradScaler); grad clipping
at 1.0; non-finite-loss skip (M-9). Each epoch logs train + **monitor** recon MSE (M-1) and best-on-
monitor / periodic / train-state checkpoints; a stage early-stops after `EARLY_STOP_PATIENCE` epochs
with no monitor improvement. A shared `train_stage()` runs both stages.

**Stage P1 — FCMAE only**, from the ImageNet-FCMAE ConvNeXtV2 checkpoint. Set `SMOKE_TEST = True` for
a quick CPU dry-run of the whole curriculum first.

In [ ]:
# ============================ Shared full-run trainer + monitoring helpers ============================
bs = 4 if SMOKE_TEST else BATCH_SIZE
nw = 0 if SMOKE_TEST else NUM_WORKERS
use_amp   = USE_AMP and DEVICE.type == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_scaler = use_amp and amp_dtype == torch.float16

# ImageNet-FCMAE reference encoder (frozen): the Gate-G4c control and the M-6 reference line.
imagenet_ref_enc = FCMAEEncoder(pretrained=PRETRAINED).to(DEVICE).eval()
for p in imagenet_ref_enc.parameters():
    p.requires_grad = False

@torch.no_grad()
def monitor_recon_mse(enc, dec, n=None):
    if len(monitor_index) == 0:
        return float("nan")
    ds = FCMAEAnchorDataset(monitor_index)
    n = n or min(2 * bs, len(ds))
    idx = np.random.default_rng(0).choice(len(ds), size=min(n, len(ds)), replace=False)
    imgs = torch.stack([ds[i][0] for i in idx])
    enc.eval(); dec.eval()
    loss, _ = run_fcmae_batch(enc, dec, imgs, augment=False)
    enc.train(); dec.train()
    return loss.item()

def train_stage(name, params, dataloader, step_fn, monitor_fn, epochs,
                best_ckpt, trainstate, history_csv, resume=None):
    """Generic warmup→cosine AMP trainer with best-on-monitor + periodic + resumable checkpointing."""
    opt = torch.optim.AdamW(params, lr=LR * (BATCH_SIZE * ACCUM_STEPS / 256), betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
    total = SMOKE_EPOCHS if SMOKE_TEST else epochs
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_factor(e):
        if e < warm:
            return (e + 1) / warm
        prog = (e - warm) / max(1, total - warm)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    start, best, since_best, history = 0, float("inf"), 0, []
    if resume and Path(resume).exists():
        ck = torch.load(resume, map_location=DEVICE)
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        start = ck["epoch"] + 1; best = ck.get("best", float("inf"))
        print(f"[{name}] resumed at epoch {start}")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    t0 = time.time()
    for epoch in range(start, total):
        running, cnt = 0.0, 0
        opt.zero_grad(set_to_none=True); micro = 0
        for step, batch in enumerate(dataloader):
            if SMOKE_TEST and step >= SMOKE_STEPS:
                break
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
                loss = step_fn(batch)
            if not torch.isfinite(loss):                 # non-finite-loss skip (M-9)
                continue
            running += loss.item(); cnt += 1            # report the unscaled per-micro-batch loss
            lb = loss / ACCUM_STEPS                      # scale so accumulated grads == a batch of BATCH_SIZE*ACCUM_STEPS
            (scaler.scale(lb) if use_scaler else lb).backward()
            micro += 1
            if micro % ACCUM_STEPS == 0:                 # optimizer step on the accumulation boundary
                if use_scaler:
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                    scaler.step(opt); scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()
                opt.zero_grad(set_to_none=True)
        if micro % ACCUM_STEPS != 0:                     # flush a trailing partial window
            if use_scaler:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()
            opt.zero_grad(set_to_none=True)
        sched.step()
        mon = monitor_fn()
        history.append(dict(epoch=epoch, train=running / max(1, cnt), monitor=mon,
                            lr=opt.param_groups[0]["lr"], secs=round(time.time() - t0, 1)))
        pd.DataFrame(history).to_csv(history_csv, index=False)
        torch.save({"epoch": epoch, "opt": opt.state_dict(), "sched": sched.state_dict(), "best": best},
                   trainstate)
        improved = mon < best
        if improved:
            best = mon; since_best = 0
            torch.save(fcmae_enc.backbone.state_dict(), best_ckpt)   # best-on-monitor encoder
        else:
            since_best += 1
        if epoch % CKPT_EVERY == 0:
            torch.save(fcmae_enc.backbone.state_dict(),
                       best_ckpt.with_name(best_ckpt.stem + ("_epoch%03d.pth" % epoch)))
        print(f"[{name}] epoch {epoch:03d}/{total} | train {history[-1]['train']:.4f} | "
              f"monitor {mon:.4f} | lr {history[-1]['lr']:.2e} | best {best:.4f}"
              + ("  *" if improved else ""))
        if since_best >= EARLY_STOP_PATIENCE:
            print(f"[{name}] early stop: no monitor improvement in {EARLY_STOP_PATIENCE} epochs"); break
    return best

# --- models (shared encoder across P1/P2), initialized from ImageNet FCMAE ---
fcmae_enc = FCMAEEncoder(pretrained=PRETRAINED).to(DEVICE)
fcmae_dec = TinyFCMAEDecoder().to(DEVICE)
if USE_GRAD_CKPT:
    try:
        fcmae_enc.backbone.set_grad_checkpointing(True); print("encoder gradient checkpointing: ON")
    except Exception as e:
        print("[warn] grad checkpointing unavailable (%s)" % type(e).__name__)

# ---------- Stage P1: FCMAE only ----------
if RUN_P1:
    p1_dl = DataLoader(FCMAEAnchorDataset(grad_index), batch_size=bs, shuffle=True, num_workers=nw,
                       drop_last=True, pin_memory=(DEVICE.type == "cuda"))
    print(f"P1 anchors: {len(p1_dl.dataset)} | batch {bs} | steps/epoch {len(p1_dl)}")
    def p1_step(batch):
        loss, _ = run_fcmae_batch(fcmae_enc, fcmae_dec, batch[0], augment=True)
        return loss
    best_p1 = train_stage("P1", list(fcmae_enc.parameters()) + list(fcmae_dec.parameters()),
                          p1_dl, p1_step, lambda: monitor_recon_mse(fcmae_enc, fcmae_dec),
                          P1_EPOCHS, FCMAE_P1_CKPT, P1_TRAINSTATE, P1_HISTORY_CSV, resume=RESUME_P1)
    if FCMAE_P1_CKPT.exists():
        fcmae_enc.backbone.load_state_dict(torch.load(FCMAE_P1_CKPT, map_location=DEVICE))   # load best-on-monitor
    print(f"P1 done | best monitor MSE {best_p1:.4f} | saved {FCMAE_P1_CKPT.name}")

**Stage P2 — joint FCMAE + cross-view completion**, initialized from the P1 encoder. `L = L_FCMAE +
λ·L_crossview`; row-wise cross-attention (AP & LAT share the vertical axis). The **pairing guards**
(§3.4) are mandatory: a static join check + the behavioural **shuffled-pair gap (M-7)** — a silent A/B
mismatch would train wrong correspondences while the loss still falls.

In [ ]:
# ---------- Stage P2: joint FCMAE + cross-view completion ----------
xview_dec = CrossViewDecoder().to(DEVICE)
ATTN_MASK = rowwise_attn_mask()

# static pairing check (§3.4.1): both views of a sampled instance share the same case/side dir.
_chk = paired_index.sample(min(8, len(paired_index)), random_state=SEED)
for _r in _chk.itertuples(index=False):
    if _r.variant == "normal":
        assert Path(_r.ap).parent == Path(_r.lat).parent, f"AP/LAT not same instance dir: {_r.case}/{_r.side}"
print(f"pairing static check PASS on {len(_chk)} instances (§3.4.1) | attn mask blocked "
      f"{(ATTN_MASK == float('-inf')).float().mean().item():.2f}")

@torch.no_grad()
def shuffled_pair_gap(enc, xdec, n=None):
    """M-7: L_crossview(shuffled source) − L_crossview(true source) on the monitor set. Want >> 0."""
    if len(monitor_index) < 2:
        return float("nan")
    ds = FCMAEPairedDataset(monitor_index)
    n = n or min(2 * bs, len(ds))
    idx = np.random.default_rng(1).choice(len(ds), size=min(n, len(ds)), replace=False)
    ap = torch.stack([ds[i][0] for i in idx]); lat = torch.stack([ds[i][1] for i in idx])
    enc.eval(); xdec.eval()
    l_true = crossview_loss(enc, xdec, ap, lat, ATTN_MASK).item()
    perm = torch.randperm(len(idx)); l_shuf = crossview_loss(enc, xdec, ap, lat[perm], ATTN_MASK).item()
    enc.train(); xdec.train()
    return l_shuf - l_true

if RUN_P2:
    if FCMAE_P1_CKPT.exists():                          # P2 starts from the P1 encoder (§3.3 curriculum)
        fcmae_enc.backbone.load_state_dict(torch.load(FCMAE_P1_CKPT, map_location=DEVICE))
    p2_dl = DataLoader(FCMAEPairedDataset(grad_index), batch_size=bs, shuffle=True, num_workers=nw,
                       drop_last=True, pin_memory=(DEVICE.type == "cuda"))
    print(f"P2 pairs: {len(p2_dl.dataset)} | batch {bs} | steps/epoch {len(p2_dl)}")
    def p2_step(batch):
        ap, lat = batch[0], batch[1]
        aps = torch.stack([apply_aug(ap[b], sample_aug_params()) for b in range(ap.shape[0])])
        lats = torch.stack([apply_aug(lat[b], sample_aug_params()) for b in range(lat.shape[0])])
        tgt, src = (aps, lats) if random.random() < 0.5 else (lats, aps)   # symmetric A↔B coin flip
        l_fcmae, _ = run_fcmae_batch(fcmae_enc, fcmae_dec, tgt, augment=False)
        return l_fcmae + LAMBDA_XVIEW * crossview_loss(fcmae_enc, xview_dec, tgt, src, ATTN_MASK)
    best_p2 = train_stage("P2", list(fcmae_enc.parameters()) + list(fcmae_dec.parameters()) + list(xview_dec.parameters()),
                          p2_dl, p2_step, lambda: monitor_recon_mse(fcmae_enc, fcmae_dec),
                          P2_EPOCHS, FCMAE_P2_CKPT, P2_TRAINSTATE, P2_HISTORY_CSV, resume=RESUME_P2)
    if FCMAE_P2_CKPT.exists():
        fcmae_enc.backbone.load_state_dict(torch.load(FCMAE_P2_CKPT, map_location=DEVICE))
    gap = shuffled_pair_gap(fcmae_enc, xview_dec)
    print(f"P2 done | best monitor MSE {best_p2:.4f} | final M-7 gap {gap:+.4f} | saved {FCMAE_P2_CKPT.name}")
    print("  M-7 read: gap ≈ 0 from the start → pairing/plumbing bug; gap shrinking → source ignored.")

## 4b. Monitoring probes (M-5 / M-6) + export / bless

Collapse detectors (M-5a per-channel std, M-5b effective rank) and the representation probe (M-6: kNN
+ linear probe on the cohort label over **frozen** features, vs the ImageNet-init reference line —
domain pretraining must beat it). Then **export / bless** (M-10, L-11): rebuild a fresh
`features_only` backbone, strict-load the chosen stage, discard every decoder/head, and write the
blessed `convnextv2_fcmae_encoder.pth` + config blob. The §5 contract validation re-runs the per-axis
variance + axis-alignment guards on this checkpoint.

In [ ]:
# --- M-5 / M-6 monitoring probes (frozen features; labels eval-only) ---
@torch.no_grad()
def pooled_embeddings(enc, df):
    ds = FCMAEAnchorDataset(df); enc.eval()
    embs, labels = [], []
    zero = None
    for i in range(len(ds)):
        img, dstag, *_ = ds[i]
        x = to_encoder_input(img.unsqueeze(0).to(device))
        mg = torch.zeros(1, 1, MASK_GRID, MASK_GRID, dtype=torch.bool, device=device)
        embs.append(enc(x, mg)[-1].mean(dim=(2, 3)).squeeze(0).float().cpu())
        labels.append(dstag)
    return torch.stack(embs), labels

def effective_rank(emb):
    x = emb - emb.mean(0, keepdim=True)
    s = torch.linalg.svdvals(x); p = (s / s.sum().clamp_min(1e-12)).clamp_min(1e-12)
    return float(torch.exp(-(p * p.log()).sum()))

def knn_loo_acc(emb, labels, k=5):
    y = np.array(labels); n = len(y)
    if n <= k or len(set(labels)) < 2:
        return float("nan")
    x = F.normalize(emb, dim=1); sim = (x @ x.t()).numpy(); np.fill_diagonal(sim, -np.inf)
    return float(np.mean([pd.Series(y[np.argsort(-sim[i])[:k]]).mode().iloc[0] == y[i] for i in range(n)]))

if len(monitor_index) >= 2:
    emb_t, lab = pooled_embeddings(fcmae_enc, monitor_index)
    emb_r, _   = pooled_embeddings(imagenet_ref_enc, monitor_index)
    frac_dead = float((emb_t.std(0) < 1e-3).float().mean())
    print(f"M-5a dead-channel fraction={frac_dead:.2%} {'⚠ collapse (check GRN / LR)' if frac_dead > 0.15 else 'ok'}")
    print(f"M-5b effective rank: trained={effective_rank(emb_t):.2f}  imagenet-ref={effective_rank(emb_r):.2f}")
    print(f"M-6 kNN cohort acc: trained={knn_loo_acc(emb_t, lab):.3f}  imagenet-ref={knn_loo_acc(emb_r, lab):.3f} "
          f"(domain pretraining must beat the reference line)")
    try:
        from sklearn.linear_model import LogisticRegression
        yb = (np.array(lab) == "fractured").astype(int)
        if len(set(yb)) == 2:
            acc = LogisticRegression(max_iter=500).fit(F.normalize(emb_t, dim=1).numpy(), yb).score(
                F.normalize(emb_t, dim=1).numpy(), yb)
            print(f"M-6 linear-probe (in-sample) acc={acc:.3f}")
    except Exception as e:
        print(f"M-6 linear-probe skipped ({type(e).__name__})")

# --- export & bless (M-10 / L-11): fresh backbone strict-load, decoder/heads discarded ---
BLESSED_STAGE = "P2" if (RUN_P2 and FCMAE_P2_CKPT.exists()) else "P1"
_src = FCMAE_P2_CKPT if BLESSED_STAGE == "P2" else FCMAE_P1_CKPT
assert _src.exists(), f"no {BLESSED_STAGE} checkpoint to bless"
fresh = make_backbone(pretrained=False)
fresh.load_state_dict(torch.load(_src, map_location="cpu"), strict=True)     # raises on any key mismatch
torch.save(fresh.state_dict(), FCMAE_CKPT)                                   # bare state_dict — downstream contract
config = dict(pretext="FCMAE" + ("+crossview(P2)" if BLESSED_STAGE == "P2" else "(P1)"),
              blessed_stage=BLESSED_STAGE, backbone=BACKBONE, img_size=IMG_SIZE, feat_dims=FEAT_DIMS,
              patch=PATCH, mask_ratio=MASK_RATIO, sigma_floor=SIGMA_FLOOR, bg_quota=BG_QUOTA,
              occ_eligible=OCC_ELIGIBLE, struct_weight=STRUCT_WEIGHT, lambda_xview=LAMBDA_XVIEW,
              xview_slack=XVIEW_SLACK, channel_adaptation="replicate_1to3",
              input_norm=dict(mean=NORM_MEAN, std=NORM_STD), monitor_split_csv=MONITOR_SPLIT_CSV.name,
              monitor_instances=[f"{d}/{c}/{s}" for d, c, s in sorted(MONITOR_INSTANCES)])
with open(FCMAE_CKPT.with_suffix(".config.json"), "w") as f:
    json.dump(config, f, indent=2)
print(f"blessed {BLESSED_STAGE} → {FCMAE_CKPT.name} (+ config.json). §5 re-runs the per-axis/axis-alignment guards (M-10).")

## 5. Checkpoint contract validation (M-10)

Load the **blessed** FCMAE encoder into `BiPlanarFeatureFusion` (the class the decoder uses), confirm
a clean transfer (`missing=0`), forward one healthy and one fractured real DRR pair, and assert the
four 3D **cube** feature shapes equal the decoder contract **and** the per-axis variance guard
(Check 2) — the export-time M-10 guard re-run on the blessed weights.

In [ ]:
fusion = BiPlanarFeatureFusion(depth=16, pretrained=PRETRAINED, freeze_encoder=True).to(DEVICE).eval()
if FCMAE_CKPT.exists():
    missing, unexpected = fusion.load_pretrained_encoder(FCMAE_CKPT)
    assert len(missing) == 0, f"M-10: blessed encoder did not load cleanly (missing={len(missing)})"

def first_pair(df, dataset):
    sub = df[df.dataset == dataset]
    return sub.iloc[0] if len(sub) else None

# orthogonal lift -> CUBE features: depth == height == width == IMG_SIZE//stride at each level
expected = [(c, IMG_SIZE // s) for c, s in zip(OUT_CHANNELS, [4, 8, 16, 32])]
samples = {ds: first_pair(paired_index, ds) for ds in ["healthy", "fractured"]}
f3d = None
for ds, r in samples.items():
    if r is None:
        print("[skip] no %s pair" % ds); continue
    ap = paired_tf(load_drr(r.ap)).unsqueeze(0).to(DEVICE)
    lat = paired_tf(load_drr(r.lat)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, f3d = fusion(ap, lat)
    print("\n%s: %s %s (%s)" % (ds, r.case, r.side, r.variant))
    for i, (f, (ec, es)) in enumerate(zip(f3d, expected)):
        B, C, D, H, W = f.shape
        ok = (C == ec and D == es and H == es and W == es)
        print("  level%d: %s  expect C=%d, DxHxW=%d^3  %s" % (i, tuple(f.shape), ec, es, "OK" if ok else "MISMATCH"))
        assert ok, "feature %d shape mismatch (expected cube %d^3)" % (i, es)
print("\nAll 3D feature shapes match the decoder contract (cube features).")

# --- Check 2 / M-10: per-axis feature variance (extrusion detector) on the blessed weights ---
print("\nCheck 2 (M-10) — per-axis feature std; every axis must be > 1e-4:")
assert f3d is not None, "no features produced; cannot run Check 2"
for i, f in enumerate(f3d):
    s0, s1, s2 = f.std(dim=2).mean().item(), f.std(dim=3).mean().item(), f.std(dim=4).mean().item()
    ok = min(s0, s1, s2) > 1e-4
    print("  level%d: std[axis0]=%.4f std[axis1]=%.4f std[axis2]=%.4f  %s"
          % (i, s0, s1, s2, "OK" if ok else "FAIL — axis collapsed"))
    assert ok, "level%d collapses along an axis — the lift is extruding" % i
print("Check 2 (M-10) PASS: 3D features vary on all three axes; blessed encoder OK.")

## 6. Learning curves (M-1: train vs monitor recon MSE)

In [ ]:
# M-1: train vs monitor recon MSE per stage. Monitor rising while train falls (widening gap) =
# memorization → raise mask ratio / early-stop at the monitor minimum.
stages = [("P1", P1_HISTORY_CSV), ("P2", P2_HISTORY_CSV)]
have = [(n, p) for n, p in stages if p.exists()]
if have:
    fig, ax = plt.subplots(1, len(have), figsize=(6 * len(have), 4), squeeze=False)
    for j, (name, path) in enumerate(have):
        h = pd.read_csv(path)
        ax[0, j].plot(h.epoch, h.train, label="train")
        ax[0, j].plot(h.epoch, h.monitor, label="monitor")
        ax[0, j].set_title(f"{name}: recon MSE (M-1)"); ax[0, j].set_xlabel("epoch"); ax[0, j].legend()
    plt.tight_layout(); plt.show()
else:
    print("no history yet - run the training cells.")

## 7. Visual QA — fused features with the trained encoder

Input DRRs plus a few level-0 fused feature channels for a healthy and a fractured case. With a
fully trained encoder these should show structured, bone-aligned activations (not the near-random
maps of the untrained smoke checkpoint).

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, (ds, r) in enumerate(samples.items()):
    if r is None:
        continue
    ap = paired_tf(load_drr(r.ap)).unsqueeze(0).to(DEVICE)
    lat = paired_tf(load_drr(r.lat)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        f2d, _ = fusion(ap, lat)
    axes[row, 0].imshow(load_drr(r.ap)[0], cmap="gray"); axes[row, 0].set_title("%s %s\nAP" % (ds, r.case))
    axes[row, 1].imshow(load_drr(r.lat)[0], cmap="gray"); axes[row, 1].set_title("LAT")
    fmap = f2d[0][0].cpu()
    for j in range(3):
        axes[row, 2 + j].imshow(fmap[j], cmap="viridis"); axes[row, 2 + j].set_title("L0 fused ch%d" % j)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# --- 3D feature visual check: orthogonal mid-slices of the LIFTED 3D volume (visible non-extrusion) ---
# Complements Check 2 (M-10): if the lift were extruding, one mid-plane would be a flat band and the
# axis1 (A-P depth) montage frames would be identical. Distinct, structured planes = a true cube.
VIZ_LEVEL, VIZ_CH = 0, 0
fig, axes = plt.subplots(len(samples), 4, figsize=(14, 3.4 * len(samples)), squeeze=False)
for row, (ds, r) in enumerate(samples.items()):
    if r is None:
        for ax in axes[row]: ax.axis("off")
        continue
    ap = paired_tf(load_drr(r.ap)).unsqueeze(0).to(DEVICE)
    lat = paired_tf(load_drr(r.lat)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, f3 = fusion(ap, lat)
    vol = f3[VIZ_LEVEL][0, VIZ_CH].cpu()                 # [S,S,S] = (axis0 L-R, axis1 A-P, axis2 S-I)
    S = vol.shape[0]; mid = S // 2
    panels = [(vol[mid, :, :], f"{ds} {r.case} L{VIZ_LEVEL} ch{VIZ_CH}\nmid axis0 (L-R)"),
              (vol[:, mid, :], "mid axis1 (A-P)"),
              (vol[:, :, mid], "mid axis2 (S-I)")]
    for j, (img, ttl) in enumerate(panels):
        axes[row, j].imshow(img, cmap="viridis"); axes[row, j].set_title(ttl, fontsize=9); axes[row, j].axis("off")
    steps = torch.linspace(0, S - 1, 5).round().long()
    axes[row, 3].imshow(torch.cat([vol[:, k, :] for k in steps], dim=1), cmap="viridis")
    axes[row, 3].set_title("axis1 (A-P) slices: depth varies", fontsize=9); axes[row, 3].axis("off")
plt.tight_layout(); plt.show()
print("Flat mid-plane or identical montage frames ⇒ extruding lift (Check 2 would also FAIL).")

## 8. Next steps

1. Copy `models/convnextv2_fcmae_encoder.pth` (+ `.config.json`) to wherever the decoder runs (it is
   git-ignored). `03_decoder_pipeline.ipynb` / `02_frontend_pretrain.ipynb` load it via
   `fusion.load_pretrained_encoder(...)` and should print `missing=0`.
2. **Gate G4** (the gate that finally matters): at smoke-test scale, run the step-2 front-end + neutral
   head under four encoder inits — FCMAE-P2, FCMAE-P1, ImageNet-FCMAE-only (`PRETRAINED` backbone, no
   domain pretraining), random-init — and compare Dice **and** surface distance **split by cohort**.
   Ship the best of P2/P1 only if it beats ImageNet-only; if ImageNet-only wins, delete the
   domain-pretraining stage and document the negative result. Random-init confirms dynamic range.
3. To extend a stage, set `RESUME_P1`/`RESUME_P2` to its `fcmae_p*_trainstate.pth` and raise the epochs.
4. `FREEZE_ENCODER=False` (decoder default) fine-tunes this encoder jointly with the decoder; set
   `True` to keep it fixed and save memory.